In [1]:
import os
import glob
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 

In [2]:
import torch 
import torch.nn as nn 
import torch.optim as optim 

In [3]:




from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models 
import torchvision.transforms.functional as TF
from torchvision import transforms
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import random

In [4]:
cfg = dict(
    # dataset paths 
    train_lenses    = "dataset/train_lenses",
    train_nonlenses = "dataset/train_nonlenses",
    test_lenses     = "dataset/test_lenses",
    test_nonlenses  = "dataset/test_nonlenses",

    # model training 
    epochs        = 30,
    batch_size    = 64,
    lr            = 1e-4,
    weight_decay  = 1e-4,
    num_workers   = 4,
    seed          = 42,

    # model checkpoint
    checkpoint    = "best_model.pth",
    roc_plot      = "roc_curve.png",
    device        = "cuda" if torch.cuda.is_available() else "cpu",
)

In [5]:
# def set_seed(s):
#     random.seed(s); np.random.seed(s)
#     torch.manual_seed(s); torch.cuda.manual_seed_all(s)
 
# set_seed(CFG["seed"])
# print(f"Using device: {CFG['device']}")

In [6]:
# Device Setup 
try:
    import torch_directml
    device = torch_directml.device()
    print(f"Using device: {device}")
except Exception:
    device = torch.device('cpu')
    print("Using device: CPU")

Using device: privateuseone:0


Loading Dataset

In [7]:
class LensDataset(Dataset):
    """
    Loads .npy files of shape (3, 64, 64).
    Each file is one galaxy; label 1 = lens, 0 = non-lens.
    Augmentation is applied only when augment=True (training set).
    Normalisation uses per-channel ImageNet stats, which transfer
    reasonably well to photometric (g,r,i) images.
    """
 
    # ImageNet channel statistics (applied per-channel)
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]


    def __init__(self, lens_dir: str, nonlens_dir: str, augment: bool = False):
        self.augment = augment
 
        lens_files    = sorted(glob.glob(os.path.join(lens_dir,    "*.npy")))
        nonlens_files = sorted(glob.glob(os.path.join(nonlens_dir, "*.npy")))
 
        if not lens_files:
            raise FileNotFoundError(f"No .npy files found in {lens_dir}")
        if not nonlens_files:
            raise FileNotFoundError(f"No .npy files found in {nonlens_dir}")
 
        self.files  = lens_files + nonlens_files
        self.labels = [1] * len(lens_files) + [0] * len(nonlens_files)
 
        n_pos = len(lens_files)
        n_neg = len(nonlens_files)
        ratio = n_neg / max(n_pos, 1)
        print(f"  lenses: {n_pos}  non-lenses: {n_neg}  imbalance ratio: {ratio:.1f}:1")

    def __len__(self):
        return len(self.files)

    def _to_tensor(self, arr: np.ndarray) -> torch.Tensor:
        """
        arr shape: (3, 64, 64), dtype float32 or float64
        Returns normalised float32 tensor (3, 64, 64)
        """
        t = torch.from_numpy(arr.astype(np.float32))   # (3, H, W)
 
        # Min-max normalise to [0, 1] per channel to handle
        # arbitrary photometric units in SDSS g, r, i bands
        for c in range(t.shape[0]):
            lo, hi = t[c].min(), t[c].max()
            if hi > lo:
                t[c] = (t[c] - lo) / (hi - lo)
            # else leave as zero
 
        # Apply ImageNet statistics
        mean = torch.tensor(self.MEAN).view(3, 1, 1)
        std  = torch.tensor(self.STD ).view(3, 1, 1)
        t = (t - mean) / std
 
        return t

    
    def __getitem__(self, idx):
        arr   = np.load(self.files[idx])
        label = self.labels[idx]
        t     = self._to_tensor(arr)
 
        if self.augment:
            # Random horizontal flip
            if random.random() > 0.5:
                t = TF.hflip(t)
            # Random vertical flip
            if random.random() > 0.5:
                t = TF.vflip(t)
            # Random 90-degree rotation (0, 90, 180, 270)
            k = random.randint(0, 3)
            t = torch.rot90(t, k, dims=[1, 2])
            # Random small rotation (±20 degrees) for more variety
            angle = random.uniform(-20, 20)
            t = TF.rotate(t, angle)
 
        return t, torch.tensor(label, dtype=torch.float32)

    

Defining the model

In [8]:
def build_model(device: str) -> nn.Module:
    """
    ResNet18 pretrained on ImageNet.
    Replace final FC with a single logit output (binary classification).
    The first conv layer already accepts 3 channels which maps well to (g, r, i).
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
 
    # Freeze early layers; only fine-tune layer3, layer4, and FC
    for name, param in model.named_parameters():
        if not any(s in name for s in ["layer3", "layer4", "fc"]):
            param.requires_grad = False
 
    # Replace head
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 1)   # single logit -> BCEWithLogitsLoss
 
    return model.to(device)

Weighted Sampling - handling the imbalance in batches

In [9]:
def make_sampler(dataset: LensDataset) -> WeightedRandomSampler:
    labels  = np.array(dataset.labels)
    n_pos   = labels.sum()
    n_neg   = len(labels) - n_pos
    weight_pos = 1.0 / n_pos if n_pos > 0 else 1.0
    weight_neg = 1.0 / n_neg if n_neg > 0 else 1.0
    sample_weights = np.where(labels == 1, weight_pos, weight_neg)
    return WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True,
    )

Training Loop

In [10]:
def train(cfg: dict):
 
    # Datasets
    train_ds = LensDataset(cfg["train_lenses"], cfg["train_nonlenses"], augment=True)
    test_ds  = LensDataset(cfg["test_lenses"],  cfg["test_nonlenses"],  augment=False)
 
    # Compute pos_weight for loss: ratio of negatives to positives
    labels_arr = np.array(train_ds.labels)
    n_pos = labels_arr.sum()
    n_neg = len(labels_arr) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], device=device)
    print(f"Loss pos_weight: {pos_weight.item():.2f}")
 
    # DataLoaders
    sampler    = make_sampler(train_ds)
    train_loader = DataLoader(
        train_ds,
        batch_size  = cfg["batch_size"],
        sampler     = sampler,           # overrides shuffle
        num_workers = cfg["num_workers"],
        pin_memory  = True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size  = cfg["batch_size"],
        shuffle     = False,
        num_workers = cfg["num_workers"],
        pin_memory  = True,
    )
 
    # Model, loss, optimiser
    model     = build_model(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr           = cfg["lr"],
        weight_decay = cfg["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["epochs"]
    )
 
    best_auc = 0.0
 
    for epoch in range(1, cfg["epochs"] + 1):
        # ── Training ──
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        scheduler.step()
 
        avg_loss = running_loss / len(train_ds)
 
        # ── Validation (on test set) ──
        model.eval()
        all_probs  = []
        all_labels = []
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs = imgs.to(device)
                logits = model(imgs).squeeze(1)
                probs  = torch.sigmoid(logits).cpu().numpy()
                all_probs.extend(probs)
                all_labels.extend(labels.numpy())
 
        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        epoch_auc   = auc(fpr, tpr)
 
        print(f"Epoch {epoch:3d}/{cfg['epochs']}  "
              f"loss={avg_loss:.4f}  AUC={epoch_auc:.4f}")
 
        if epoch_auc > best_auc:
            best_auc = epoch_auc
            torch.save(model.state_dict(), cfg["checkpoint"])
            print(f"  ✓ New best AUC {best_auc:.4f} – checkpoint saved")
 
    print(f"\nTraining complete. Best AUC: {best_auc:.4f}")
    return best_auc

Final evaluation

In [11]:
def evaluate(cfg: dict):
    device = cfg["device"]
 
    test_ds = LensDataset(cfg["test_lenses"], cfg["test_nonlenses"], augment=False)
    test_loader = DataLoader(
        test_ds,
        batch_size  = cfg["batch_size"],
        shuffle     = False,
        num_workers = cfg["num_workers"],
    )
 
    # Load best checkpoint
    model = build_model(device)
    model.load_state_dict(torch.load(cfg["checkpoint"], map_location=device))
    model.eval()
 
    all_probs  = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(device)
            logits = model(imgs).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
 
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
 
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
 
    print(f"\n=== Test set AUC: {roc_auc:.4f} ===")
 
    # Plot ROC curve
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color="steelblue", lw=2,
             label=f"ResNet18 (AUC = {roc_auc:.4f})")
    plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.02])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve – Gravitational Lens Detector")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(cfg["roc_plot"], dpi=150)
    print(f"ROC curve saved to {cfg['roc_plot']}")
    plt.show()
 
    return roc_auc, fpr, tpr, thresholds

Entry Point

In [ ]:
if __name__ == "__main__":
    print("=" * 55)
    print("  Gravitational Lens Classifier – Training")
    print("=" * 55)
    train(cfg)
 
    print("\n" + "=" * 55)
    print("  Final Evaluation on Test Set")
    print("=" * 55)
    evaluate(cfg)

  Gravitational Lens Classifier – Training
  lenses: 1730  non-lenses: 28675  imbalance ratio: 16.6:1
  lenses: 195  non-lenses: 19455  imbalance ratio: 99.8:1
Loss pos_weight: 16.58
